# WxBS — Top-5 & Bottom-5 Matches with GT Error

For each of 4 WxBS scenes (kremlin, vatutin, bridge, petrzin):
- Run **outdoor-large-LA** to get predicted matches + confidence scores
- Select the **top-5** (highest conf) and **bottom-5** (lowest conf) matches
- For each match, find the nearest GT correspondence in image0, use its paired GT point in image1 as the ground truth target
- Draw both images side by side with match lines coloured green (top) / red (bottom), show pixel error to GT

In [ ]:
import os, sys
import importlib.util
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from scipy.spatial.distance import cdist

ROOT = '/Users/siddharthraj/classes/cv/cv_final/MatchFormer'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Load defaultmf directly — avoids clash with the 'config' pip package in this venv
_spec = importlib.util.spec_from_file_location(
    'defaultmf', os.path.join(ROOT, 'config', 'defaultmf.py'))
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
get_cfg_defaults = _mod.get_cfg_defaults

from model.lightning_loftr import PL_LoFTR

print('Imports OK')

## Config

In [ ]:
WXBS_ROOT = '/Users/siddharthraj/classes/cv/cv_final/data/WxBS_data_folder/v1.1'
CKPT      = os.path.join(ROOT, 'model/weights/outdoor-large-LA.ckpt')
OUT_DIR   = os.path.join(ROOT, 'phase2/visualizations/wxbs_top_bottom')
os.makedirs(OUT_DIR, exist_ok=True)

SCENES = [
    ('WGABS', 'kremlin'),
    ('WGABS', 'vatutin'),
    ('WGALBS', 'bridge'),
    ('WGABS',  'petrzin'),
]

TARGET_H, TARGET_W = 480, 640
TOP_K = 5

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'CKPT exists: {os.path.isfile(CKPT)}')

## Load Model

In [ ]:
config = get_cfg_defaults()
config.MATCHFORMER.BACKBONE_TYPE = 'largela'
config.MATCHFORMER.SCENS = 'outdoor'
config.MATCHFORMER.RESOLUTION = (8, 2)
config.MATCHFORMER.COARSE.D_MODEL = 256
config.MATCHFORMER.COARSE.D_FFN = 256
model = PL_LoFTR(config, pretrained_ckpt=CKPT).to(device).eval()
print('Model loaded')

## Quick Test — One Scene, One Bad Match

In [ ]:
from scipy.spatial.distance import cdist

TARGET_H, TARGET_W = 480, 640

def find_image(scene_dir, stem):
    for ext in ('.jpg', '.png'):
        p = os.path.join(scene_dir, stem + ext)
        if os.path.isfile(p): return p

def load_gray_tensor(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    oh, ow = img.shape
    img_r = cv2.resize(img, (TARGET_W, TARGET_H))
    t = torch.from_numpy(img_r.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0)
    return t, oh, ow

def load_color(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (TARGET_W, TARGET_H))

# ── Pick one scene ────────────────────────────────────────────────────────────
WXBS_ROOT = '/Users/siddharthraj/classes/cv/cv_final/data/WxBS_data_folder/v1.1'
scene_dir = os.path.join(WXBS_ROOT, 'WGABS', 'kremlin')

p0, p1 = find_image(scene_dir, '01'), find_image(scene_dir, '02')
t0, oh0, ow0 = load_gray_tensor(p0)
t1, oh1, ow1 = load_gray_tensor(p1)

# Load & scale GT correspondences
corrs = np.loadtxt(os.path.join(scene_dir, 'corrs.txt'))
gt_pts0 = corrs[:, :2] * [TARGET_W/ow0, TARGET_H/oh0]
gt_pts1 = corrs[:, 2:] * [TARGET_W/ow1, TARGET_H/oh1]

# ── Run model ─────────────────────────────────────────────────────────────────
model.matcher.coarse_matching.epipolar_F = None
model.matcher.coarse_matching.thr = 0.2
data = {'image0': t0.to(device), 'image1': t1.to(device)}
with torch.no_grad():
    model.matcher(data)

mkpts0 = data['mkpts0_f'].cpu().numpy()
mkpts1 = data['mkpts1_f'].cpu().numpy()
mconf  = data['mconf'].cpu().numpy()
print(f'Predicted matches: {len(mkpts0)}')

# ── Find worst match by GT error ──────────────────────────────────────────────
dists   = cdist(mkpts0, gt_pts0)
nn_idx  = np.argmin(dists, axis=1)
gt_tgt1 = gt_pts1[nn_idx]
errors  = np.linalg.norm(mkpts1 - gt_tgt1, axis=1)

worst   = np.argmax(errors)
print(f'Worst match idx: {worst}  conf={mconf[worst]:.3f}  error={errors[worst]:.1f}px')
print(f'  pred0 = ({mkpts0[worst,0]:.1f}, {mkpts0[worst,1]:.1f})')
print(f'  pred1 = ({mkpts1[worst,0]:.1f}, {mkpts1[worst,1]:.1f})')
print(f'  GT1   = ({gt_tgt1[worst,0]:.1f}, {gt_tgt1[worst,1]:.1f})')

# ── Visualize ─────────────────────────────────────────────────────────────────
img0 = load_color(p0)
img1 = load_color(p1)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'WGABS/kremlin — worst match  |  conf={mconf[worst]:.3f}  err={errors[worst]:.1f}px', fontsize=12)

ax0.imshow(img0)
ax0.scatter(gt_pts0[:,0], gt_pts0[:,1], s=10, c='yellow', alpha=0.5, label='GT pts')
ax0.plot(*mkpts0[worst], 'o', color='cyan', markersize=10, markeredgecolor='white', markeredgewidth=1, label='Query')
ax0.set_title('Image 0 — query point (cyan)', fontsize=10)
ax0.legend(fontsize=8, loc='upper right')
ax0.axis('off')

ax1.imshow(img1)
ax1.scatter(gt_pts1[:,0], gt_pts1[:,1], s=10, c='yellow', alpha=0.5, label='GT pts')
ax1.plot(*mkpts1[worst],    'o', color='red',     markersize=10, markeredgecolor='white', markeredgewidth=1, label=f'Pred')
ax1.plot(*gt_tgt1[worst],   '+', color='lime',    markersize=14, markeredgewidth=2.5,                        label=f'GT target')
ax1.plot([mkpts1[worst,0], gt_tgt1[worst,0]],
         [mkpts1[worst,1], gt_tgt1[worst,1]], 'w--', lw=1.5, label=f'Error={errors[worst]:.1f}px')
ax1.set_title('Image 1 — red=pred, green+=GT, dashed=error', fontsize=10)
ax1.legend(fontsize=8, loc='upper right')
ax1.axis('off')

plt.tight_layout()
plt.show()

## Helper Functions

In [ ]:
def find_image(scene_dir, stem):
    for ext in ('.jpg', '.png'):
        p = os.path.join(scene_dir, stem + ext)
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f'No {stem}.jpg/.png in {scene_dir}')


def load_gray_tensor(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    orig_h, orig_w = img.shape
    img_r = cv2.resize(img, (TARGET_W, TARGET_H))
    t = torch.from_numpy(img_r.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0)
    return t, orig_h, orig_w


def load_color(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (TARGET_W, TARGET_H))


def run_model(scene_dir):
    """Run model on a WxBS scene, return mkpts0, mkpts1, mconf + scale info."""
    p0 = find_image(scene_dir, '01')
    p1 = find_image(scene_dir, '02')

    t0, oh0, ow0 = load_gray_tensor(p0)
    t1, oh1, ow1 = load_gray_tensor(p1)

    # Scale GT coords to resized image
    corrs = np.loadtxt(os.path.join(scene_dir, 'corrs.txt'))
    if corrs.ndim == 1:
        corrs = corrs.reshape(1, -1)
    gt_pts0 = corrs[:, :2] * np.array([TARGET_W / ow0, TARGET_H / oh0])
    gt_pts1 = corrs[:, 2:] * np.array([TARGET_W / ow1, TARGET_H / oh1])

    model.matcher.coarse_matching.epipolar_F = None
    model.matcher.coarse_matching.thr = 0.2
    data = {'image0': t0.to(device), 'image1': t1.to(device)}
    with torch.no_grad():
        model.matcher(data)

    mkpts0 = data['mkpts0_f'].cpu().numpy()
    mkpts1 = data['mkpts1_f'].cpu().numpy()
    mconf  = data['mconf'].cpu().numpy()

    img0_color = load_color(p0)
    img1_color = load_color(p1)

    return mkpts0, mkpts1, mconf, gt_pts0, gt_pts1, img0_color, img1_color


def nearest_gt_error(pred0, pred1, gt_pts0, gt_pts1):
    """
    For each predicted match (pred0[i], pred1[i]):
      - Find nearest GT point in image0
      - GT target in image1 is the paired GT point
      - Pixel error = ||pred1[i] - gt_pts1[nearest]||
    Returns: errors [N], gt_targets [N,2], nn_indices [N]
    """
    if len(pred0) == 0 or len(gt_pts0) == 0:
        return np.array([]), np.zeros((0,2)), np.array([])
    dists = cdist(pred0, gt_pts0)           # [N_pred, N_gt]
    nn_idx = np.argmin(dists, axis=1)       # nearest GT in img0 for each pred
    gt_target = gt_pts1[nn_idx]             # paired GT in img1
    errors = np.linalg.norm(pred1 - gt_target, axis=1)
    return errors, gt_target, nn_idx

## Visualization Function

In [ ]:
def draw_matches(ax0, ax1, pts0, pts1, gt_targets1, errors, confs, color, label_prefix, fig):
    """Draw match lines + GT targets on ax0/ax1 using matplotlib."""
    for i, (p0, p1, gt1, err, conf) in enumerate(zip(pts0, pts1, gt_targets1, errors, confs)):
        rank = i + 1

        # Point in image0
        ax0.plot(*p0, 'o', color=color, markersize=7,
                 markeredgecolor='white', markeredgewidth=0.8)
        ax0.text(p0[0]+4, p0[1]-4, f'{label_prefix}{rank}',
                 color=color, fontsize=7, fontweight='bold')

        # Predicted point in image1
        ax1.plot(*p1, 'o', color=color, markersize=7,
                 markeredgecolor='white', markeredgewidth=0.8)

        # GT target in image1 (white X)
        ax1.plot(*gt1, '+', color='white', markersize=9, markeredgewidth=1.5)

        # Line from pred to GT in image1
        ax1.plot([p1[0], gt1[0]], [p1[1], gt1[1]], '-', color='white', lw=0.8, alpha=0.7)

        # Error label
        ax1.text(p1[0]+4, p1[1]-4,
                 f'{label_prefix}{rank}\nerr={err:.1f}px\nc={conf:.3f}',
                 color=color, fontsize=6.5, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.55))


def visualize_scene(cat, scene_name):
    scene_dir = os.path.join(WXBS_ROOT, cat, scene_name)
    mkpts0, mkpts1, mconf, gt_pts0, gt_pts1, img0, img1 = run_model(scene_dir)

    n = len(mkpts0)
    print(f'\n{cat}/{scene_name}: {n} matches  |  {len(gt_pts0)} GT points')

    if n == 0:
        print('  No matches — skipping')
        return

    # Compute GT errors for all matches
    errors, gt_targets1, nn_idx = nearest_gt_error(mkpts0, mkpts1, gt_pts0, gt_pts1)

    # Top-K and Bottom-K by confidence
    sorted_by_conf = np.argsort(mconf)[::-1]
    top_idx    = sorted_by_conf[:TOP_K]
    bottom_idx = sorted_by_conf[max(0, n-TOP_K):][::-1]   # lowest conf first

    print(f'  Top-5    conf range: [{mconf[top_idx[-1]]:.3f}, {mconf[top_idx[0]]:.3f}]')
    print(f'  Bottom-5 conf range: [{mconf[bottom_idx[-1]]:.3f}, {mconf[bottom_idx[0]]:.3f}]')
    print(f'\n  {"Rank":<6} {"Type":<8} {"Conf":>6} {"Err (px)":>10}  pt0          pt1          GT1')
    print(f'  {"-"*80}')
    for rank, idx in enumerate(top_idx, 1):
        p0, p1, gt1 = mkpts0[idx], mkpts1[idx], gt_targets1[idx]
        print(f'  #{rank:<5} {"TOP":<8} {mconf[idx]:>6.3f} {errors[idx]:>9.1f}px'
              f'  ({p0[0]:5.1f},{p0[1]:5.1f})  ({p1[0]:5.1f},{p1[1]:5.1f})  ({gt1[0]:5.1f},{gt1[1]:5.1f})')
    for rank, idx in enumerate(bottom_idx, 1):
        p0, p1, gt1 = mkpts0[idx], mkpts1[idx], gt_targets1[idx]
        print(f'  #{rank:<5} {"BOTTOM":<8} {mconf[idx]:>6.3f} {errors[idx]:>9.1f}px'
              f'  ({p0[0]:5.1f},{p0[1]:5.1f})  ({p1[0]:5.1f},{p1[1]:5.1f})  ({gt1[0]:5.1f},{gt1[1]:5.1f})')

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(
        f'{cat}/{scene_name}  |  {n} total matches  |  {len(gt_pts0)} GT correspondences',
        fontsize=13, fontweight='bold'
    )

    for row, (label, indices, color, title) in enumerate([
        ('T', top_idx,    '#00e676', f'Top-{TOP_K} matches (highest confidence)'),
        ('B', bottom_idx, '#ff1744', f'Bottom-{TOP_K} matches (lowest confidence)'),
    ]):
        ax0, ax1 = axes[row]

        ax0.imshow(img0)
        ax0.set_title(f'Image 0 — {title}', fontsize=9)
        ax0.axis('off')

        ax1.imshow(img1)
        ax1.set_title(f'Image 1 — pred (circle) + GT (white +), white line = error', fontsize=9)
        ax1.axis('off')

        # All GT points (faint)
        ax0.scatter(gt_pts0[:,0], gt_pts0[:,1], s=8, c='yellow', alpha=0.4, linewidths=0)
        ax1.scatter(gt_pts1[:,0], gt_pts1[:,1], s=8, c='yellow', alpha=0.4, linewidths=0)

        draw_matches(ax0, ax1,
                     mkpts0[indices], mkpts1[indices],
                     gt_targets1[indices], errors[indices],
                     mconf[indices], color, label, fig)

        # Conf + error summary in corner
        summary = '\n'.join([
            f'{label}{r+1}: conf={mconf[idx]:.3f}  err={errors[idx]:.1f}px'
            for r, idx in enumerate(indices)
        ])
        ax1.text(4, TARGET_H - 4, summary, va='bottom', ha='left',
                 fontsize=6.5, color='white', family='monospace',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.6))

    legend = [
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#00e676', markersize=8, label='Top-5 pred'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff1744', markersize=8, label='Bottom-5 pred'),
        Line2D([0],[0], marker='+', color='white', markersize=8, markeredgewidth=1.5, label='GT target (img1)'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor='yellow', markersize=6, alpha=0.5, label='All GT pts'),
    ]
    fig.legend(handles=legend, loc='lower center', ncol=4, fontsize=9, framealpha=0.7)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    out_path = os.path.join(OUT_DIR, f'{cat}_{scene_name}.png')
    plt.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'\n  Saved: {out_path}')

## Run All Scenes

In [ ]:
# for cat, scene in SCENES:
#     visualize_scene(cat, scene)

## Coarse Confidence Drill-Down — Worst Match

Pick the match with the highest reprojection error from a scene.
For that bad query point in image 0, visualize the **top-5 coarse grid cells** the model considered in image 1,
overlaid with the GT target and predicted location.

In [ ]:
COARSE_STRIDE = 8
H_C, W_C = TARGET_H // COARSE_STRIDE, TARGET_W // COARSE_STRIDE  # 60, 80
MATCH_RADIUS = 20.0  # max px to consider a prediction as matching a GT point

def pixel_to_coarse_idx(px, py):
    cx = min(int(px) // COARSE_STRIDE, W_C - 1)
    cy = min(int(py) // COARSE_STRIDE, H_C - 1)
    return cy * W_C + cx

def coarse_idx_to_pixel(idx):
    cx = (idx % W_C) * COARSE_STRIDE + COARSE_STRIDE // 2
    cy = (idx // W_C) * COARSE_STRIDE + COARSE_STRIDE // 2
    return int(cx), int(cy)

# ── Pick a scene ──────────────────────────────────────────────────────────────
SCENE_CAT, SCENE_NAME = 'WGABS', 'kremlin'
scene_dir = os.path.join(WXBS_ROOT, SCENE_CAT, SCENE_NAME)

p0_path = find_image(scene_dir, '01')
p1_path = find_image(scene_dir, '02')
t0, oh0, ow0 = load_gray_tensor(p0_path)
t1, oh1, ow1 = load_gray_tensor(p1_path)

corrs = np.loadtxt(os.path.join(scene_dir, 'corrs.txt'))
if corrs.ndim == 1:
    corrs = corrs.reshape(1, -1)
gt_pts0 = corrs[:, :2] * np.array([TARGET_W / ow0, TARGET_H / oh0])
gt_pts1 = corrs[:, 2:] * np.array([TARGET_W / ow1, TARGET_H / oh1])

# ── Run model & keep conf_matrix ─────────────────────────────────────────────
model.matcher.coarse_matching.epipolar_F = None
model.matcher.coarse_matching.thr = 0.2
data = {'image0': t0.to(device), 'image1': t1.to(device)}
with torch.no_grad():
    model.matcher(data)

mkpts0 = data['mkpts0_f'].cpu().numpy()
mkpts1 = data['mkpts1_f'].cpu().numpy()
mconf  = data['mconf'].cpu().numpy()
conf_matrix = data['conf_matrix'][0].cpu().numpy()  # [L, S]

img0 = load_color(p0_path)
img1 = load_color(p1_path)

# ── Fundamental matrix from GT correspondences ───────────────────────────────
F_mat, mask = cv2.findFundamentalMat(gt_pts0, gt_pts1, cv2.FM_RANSAC, 3.0)
print(f'F matrix estimated from {int(mask.sum())}/{len(gt_pts0)} GT inliers')

# ── For each GT point, find nearest predicted match within radius ─────────────
dists = cdist(gt_pts0, mkpts0)             # [N_gt, N_pred]
nn_pred_idx = np.argmin(dists, axis=1)     # nearest prediction for each GT
nn_pred_dist = dists[np.arange(len(gt_pts0)), nn_pred_idx]

has_match = nn_pred_dist < MATCH_RADIUS
print(f'GT points with a prediction within {MATCH_RADIUS}px: {has_match.sum()}/{len(gt_pts0)}')
assert has_match.any(), f'No GT point has a prediction within {MATCH_RADIUS}px'

# Error = distance between predicted img1 location and GT img1 location
matched_gt_idx = np.where(has_match)[0]
matched_pred_idx = nn_pred_idx[has_match]
errors = np.linalg.norm(mkpts1[matched_pred_idx] - gt_pts1[matched_gt_idx], axis=1)

# Pick the worst
worst_local = np.argmax(errors)
gi = matched_gt_idx[worst_local]    # index into gt_pts0/gt_pts1
pi = matched_pred_idx[worst_local]  # index into mkpts0/mkpts1

# Query = GT point (so epipolar line passes through GT in img1)
query_pt0 = gt_pts0[gi]
gt_pt1    = gt_pts1[gi]
pred_pt1  = mkpts1[pi]

print(f'\nWorst GT-matched point (err={errors[worst_local]:.1f}px, conf={mconf[pi]:.4f}):')
print(f'  GT     (img0): ({query_pt0[0]:.1f}, {query_pt0[1]:.1f})')
print(f'  GT     (img1): ({gt_pt1[0]:.1f}, {gt_pt1[1]:.1f})')
print(f'  Pred   (img1): ({pred_pt1[0]:.1f}, {pred_pt1[1]:.1f})')

# ── Top-1 coarse candidate for this GT query point ───────────────────────────
q_idx = pixel_to_coarse_idx(query_pt0[0], query_pt0[1])
conf_row = conf_matrix[q_idx, :]
top1_idx = np.argmax(conf_row)
top1_conf = conf_row[top1_idx]
top1_px, top1_py = coarse_idx_to_pixel(top1_idx)
print(f'Top-1 coarse match: pixel=({top1_px},{top1_py})  conf={top1_conf:.6f}')

# ── Epipolar line for the GT query point ──────────────────────────────────────
p0_h = np.array([query_pt0[0], query_pt0[1], 1.0])
a, b, c_epi = F_mat @ p0_h

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'{SCENE_CAT}/{SCENE_NAME} — Worst GT-matched point (err={errors[worst_local]:.1f}px)',
    fontsize=13, fontweight='bold')

# Image 0: GT query point
ax0.imshow(img0)
ax0.scatter(gt_pts0[:,0], gt_pts0[:,1], s=8, c='yellow', alpha=0.4, label='All GT')
ax0.plot(query_pt0[0], query_pt0[1], 'o', color='cyan', markersize=12,
         markeredgecolor='white', markeredgewidth=1.5, label='Query (GT)')
cell_x = (int(query_pt0[0]) // COARSE_STRIDE) * COARSE_STRIDE
cell_y = (int(query_pt0[1]) // COARSE_STRIDE) * COARSE_STRIDE
ax0.add_patch(patches.Rectangle((cell_x, cell_y), COARSE_STRIDE, COARSE_STRIDE,
              linewidth=2, edgecolor='cyan', facecolor='none'))
ax0.set_title('Image 0 — GT query point (cyan)', fontsize=10)
ax0.legend(fontsize=8, loc='upper right')
ax0.axis('off')

# Image 1: top-1 coarse + pred + GT + epipolar
ax1.imshow(img1)
ax1.scatter(gt_pts1[:,0], gt_pts1[:,1], s=8, c='yellow', alpha=0.4, label='All GT')

# Epipolar line (white dashed) — should pass through GT
if abs(b) > 1e-8:
    x_vals = np.array([0, TARGET_W])
    y_vals = -(a * x_vals + c_epi) / b
    ax1.plot(x_vals, y_vals, 'w--', linewidth=2, alpha=0.8, label='Epipolar line')
elif abs(a) > 1e-8:
    x_val = -c_epi / a
    ax1.axvline(x=x_val, color='white', linestyle='--', linewidth=2, alpha=0.8, label='Epipolar line')

# Top-1 coarse (green)
rx = (top1_idx % W_C) * COARSE_STRIDE
ry = (top1_idx // W_C) * COARSE_STRIDE
ax1.add_patch(patches.Rectangle((rx, ry), COARSE_STRIDE, COARSE_STRIDE,
              linewidth=2, edgecolor='#00ff00', facecolor='#00ff00', alpha=0.3))
ax1.plot(top1_px, top1_py, 'o', color='#00ff00', markersize=10,
         markeredgecolor='white', markeredgewidth=1,
         label=f'Top-1 coarse ({top1_px},{top1_py}) c={top1_conf:.5f}')

# Predicted (red X)
ax1.plot(pred_pt1[0], pred_pt1[1], 'x', color='red', markersize=14,
         markeredgewidth=2.5, label=f'Pred ({pred_pt1[0]:.0f},{pred_pt1[1]:.0f})')

# GT (magenta X)
ax1.plot(gt_pt1[0], gt_pt1[1], 'x', color='magenta', markersize=14,
         markeredgewidth=2.5, label=f'GT ({gt_pt1[0]:.0f},{gt_pt1[1]:.0f})')

# Arrow pred -> GT
ax1.annotate('', xy=gt_pt1, xytext=pred_pt1,
             arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

ax1.set_title('Image 1 — Coarse (green) + Pred (red X) + GT (magenta X) + Epipolar (dashed)', fontsize=9)
ax1.legend(fontsize=8, loc='lower right', framealpha=0.7)
ax1.set_xlim(0, TARGET_W)
ax1.set_ylim(TARGET_H, 0)
ax1.axis('off')

plt.tight_layout()
out_path = os.path.join(OUT_DIR, f'{SCENE_CAT}_{SCENE_NAME}_coarse_drilldown.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## Coarse Confidence Heatmap on Image 1

For the same bad query point, reshape its confidence row into the 60x80 coarse grid and overlay it as a heatmap on image 1, with the epipolar line on top.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('gt_epipolar',
    os.path.join(os.getcwd(), '..', '..', 'gt_epipolar.py'))
gt_epi = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gt_epi)

TAU = 10.0

# Gaussian epipolar mask for this query point
gauss_mask = gt_epi.get_epipolar_mask(
    F_mat, H_C, W_C, H_C, W_C, tau=TAU, mode='gaussian', device='cpu'
).squeeze(0).numpy()  # [L, S]
gauss_row = gauss_mask[q_idx, :]  # [S]

# Masked confidence (log-scaled for visibility)
masked_conf = conf_row * gauss_row
masked_log = np.log(masked_conf + 1e-12).reshape(H_C, W_C)
masked_up = cv2.resize(masked_log, (TARGET_W, TARGET_H), interpolation=cv2.INTER_NEAREST)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'{SCENE_CAT}/{SCENE_NAME} — Gaussian-masked confidence (tau={TAU})',
    fontsize=13, fontweight='bold')

# ── Left: source image + query point ──
ax0.imshow(img0)
ax0.plot(query_pt0[0], query_pt0[1], 'o', color='cyan', markersize=12,
         markeredgecolor='white', markeredgewidth=1.5, label='Query (GT)')
cell_x = (int(query_pt0[0]) // COARSE_STRIDE) * COARSE_STRIDE
cell_y = (int(query_pt0[1]) // COARSE_STRIDE) * COARSE_STRIDE
ax0.add_patch(patches.Rectangle((cell_x, cell_y), COARSE_STRIDE, COARSE_STRIDE,
              linewidth=2, edgecolor='cyan', facecolor='none'))
ax0.set_title('Source — Query point', fontsize=11)
ax0.legend(fontsize=9, loc='upper right')
ax0.axis('off')

# ── Right: target image + heatmap + epipolar line + GT + top-1 ──
ax1.imshow(img1, alpha=0.3)
im = ax1.imshow(masked_up, cmap='hot', alpha=0.7)
plt.colorbar(im, ax=ax1, fraction=0.046, label='log(conf × gauss)')

# Epipolar line
if abs(b) > 1e-8:
    x_vals = np.array([0, TARGET_W])
    y_vals = -(a * x_vals + c_epi) / b
    ax1.plot(x_vals, y_vals, 'w--', lw=2, alpha=0.9, label='Epipolar line')
elif abs(a) > 1e-8:
    x_val = -c_epi / a
    ax1.axvline(x=x_val, color='white', linestyle='--', lw=2, alpha=0.9, label='Epipolar line')

# GT (magenta X)
ax1.plot(gt_pt1[0], gt_pt1[1], 'x', color='magenta', ms=14, mew=2.5,
         label=f'GT ({gt_pt1[0]:.0f},{gt_pt1[1]:.0f})')

# Top-1 after Gaussian masking (green)
top1_g = np.argmax(masked_conf)
t1x = (top1_g % W_C) * COARSE_STRIDE + COARSE_STRIDE // 2
t1y = (top1_g // W_C) * COARSE_STRIDE + COARSE_STRIDE // 2
ax1.plot(t1x, t1y, 'o', color='#00ff00', ms=10,
         markeredgecolor='white', markeredgewidth=1,
         label=f'Top-1 ({t1x},{t1y}) c={masked_conf.max():.6f}')

ax1.set_title('Target — Gaussian-masked confidence + Epipolar line', fontsize=11)
ax1.legend(fontsize=8, loc='lower right', framealpha=0.7)
ax1.set_xlim(0, TARGET_W); ax1.set_ylim(TARGET_H, 0)
ax1.axis('off')

plt.tight_layout()
out_path = os.path.join(OUT_DIR, f'{SCENE_CAT}_{SCENE_NAME}_gauss_conf_heatmap.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

dist_to_gt = np.sqrt((t1x - gt_pt1[0])**2 + (t1y - gt_pt1[1])**2)
print(f'Top-1 dist to GT: {dist_to_gt:.1f}px  |  conf range: [{masked_conf.min():.2e}, {masked_conf.max():.2e}]')